# ACHRP v5 — v4 pipeline + multi-key rotation, parallel LLM calls, separate generator model

**Flow:** `answer → atomic claims → chunks → hybrid retrieval (dense + BM25) → NLI (chunks + whole abstract + every sentence) → 18 features → [LLM judge score] → logistic regression (grouped nested CV, tuned threshold) → label → selective repair → re-verification`

**What changed vs v2**
| # | Change | Why it helps F1 / precision / recall |
|---|---|---|
| 1 | Every cell appears once (v2 had the whole pipeline twice; a patch cell overrode `judge`) | no stale outputs, no silent overrides |
| 2 | 6 new features: sentence-level NLI (`s_ent_max`, `s_ent_top2`, `s_con_max`) and polarity cues (`flips`, `neg_diff`, `hedge_novel`) | catches "often → rarely" style hallucinations that chunk-level NLI misses |
| 3 | LLM judge scores **every** answer (`JUDGE_MODE = "all"`); its P(hallucinated) is a classifier feature | v2 judged only 5/50 answers although the judge was far stronger than the NLI features |
| 4 | Decision threshold tuned for F1 **inside** the training folds (nested) | balances precision vs recall without test leakage |
| 5 | Record-level bootstrap CI for accuracy and F1 | a record's two answers move together |
| 6 | GPU-aware defaults (DeBERTa-large + 250 records on GPU, base + 60 records on CPU) | CPU ran ~100 s per record |
| 7 | Judge preflight, retries, resumable caches, `judge_p` retry with more tokens | one bad key / empty reply no longer kills a long run |

Run the cells top to bottom. Cell 8 (smoke test) is optional. Cell 9 is checkpointed and can be stopped / re-run.

**v5 changes vs v4:** judge stays `openai/gpt-oss-120b`; the answer generator is `llama-3.3-70b-versatile` (different model family, so no self-judging); `N_GEN = 250`; several Groq keys (`GROQ_API_KEYS` in `.env`, from DIFFERENT accounts) are rotated automatically and LLM calls run in parallel; hard-coded key cell removed; old oracle repair (Step 4) is OFF.

**Setup:** put `GROQ_API_KEYS=gsk_a,gsk_b,gsk_c,gsk_d` in your `.env`, run Cell 0 first.


In [1]:
# CELL 0 — load the Groq key from your .env file (does not print the key)
import os
from pathlib import Path

def load_env(path):
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.replace("export ", "").strip()
        os.environ[k] = v.strip().strip('"').strip("'")   # overrides any stale value

candidates = [Path.cwd() / ".env", Path(r"D:\FYP\code") / ".env",
              Path.cwd().parent / ".env"]
env_file = next((p for p in candidates if p.exists()), None)

if env_file is None:
    print("No .env found. Looked in:", *[str(p) for p in candidates], sep="\n  ")
else:
    load_env(env_file)
    ks = [k.strip() for k in (os.getenv("GROQ_API_KEYS") or "").split(",") if k.strip().startswith("gsk_")]
    print("Loaded:", env_file)
    print("Keys in GROQ_API_KEYS:", len(ks), "(use keys from DIFFERENT Groq accounts)")

Loaded: c:\Users\Huawei\OneDrive\Desktop\New folder - Copy\.env
Keys in GROQ_API_KEYS: 5 (use keys from DIFFERENT Groq accounts)


In [2]:
# CELL 1 — INSTALL REQUIRED PACKAGES (only installs what is missing)
import sys, subprocess, importlib.util

PACKAGES = [("numpy", "numpy"), ("pandas", "pandas"), ("requests", "requests"), ("torch", "torch"),
            ("tqdm", "tqdm"), ("scikit-learn", "sklearn"), ("sentence-transformers", "sentence_transformers"),
            ("spacy", "spacy"), ("sentencepiece", "sentencepiece"), ("openpyxl", "openpyxl")]

for package, import_name in PACKAGES:
    if importlib.util.find_spec(import_name) is None:
        print("Installing:", package)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", package])
print("All packages are ready.")


All packages are ready.


In [3]:
# ============================================================
# CELL 2 — CONFIGURATION (v3)
# ============================================================
from pathlib import Path
import os
import torch

# ---------------- paths ----------------
_WIN_DIR = Path(r"D:\FYP\code")
PROJECT_DIR = _WIN_DIR if (os.name == "nt" and _WIN_DIR.exists()) else Path.cwd()
DATA_PATH_OVERRIDE = None      # e.g. r"C:\path\to\_pqa_labeled_train.csv.xlsx"   (None = search automatically)

# ---------------- hardware-aware defaults ----------------
HAS_GPU     = torch.cuda.is_available()
N_RECORDS   = None             # None = auto (250 records on GPU, 60 on CPU). Or set an integer, e.g. 250.
RANDOM_SEED = 42
if N_RECORDS is None:
    N_RECORDS = 250 if HAS_GPU else 250

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
NLI_MODEL   = None             # None = auto: DeBERTa-v3-LARGE on GPU, DeBERTa-v3-BASE on CPU
if NLI_MODEL is None:
    NLI_MODEL = ("MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli" if HAS_GPU
                 else "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli")

# ---------------- chunking + retrieval ----------------
CHUNK_SIZE, CHUNK_OVERLAP = 3, 1     # 3-sentence windows, 1 overlap
TOP_K = 2                            # chunks per claim (hybrid dense + BM25, RRF)
USE_FULL_CONTEXT = True              # also test the claim against the WHOLE abstract ...
FULL_CONTEXT_IF_ENTAIL_BELOW = 0.70  # ... but only when the cheap chunk pass did not already entail it
MAX_PREMISE_WORDS = 380              # keeps premise + claim inside the 512-token NLI window

# ---------------- claim status thresholds (used for claim status labels + repair only) ----------------
ENTAILMENT_THRESHOLD    = 0.50
CONTRADICTION_THRESHOLD = 0.50
SEMANTIC_THRESHOLD      = 0.30

# ---------------- LLM (judge + repair): any OpenAI-compatible endpoint ----------------
# Groq   (keys start with "gsk_"): https://api.groq.com/openai/v1/chat/completions
# xAI    (keys start with "xai-"): https://api.x.ai/v1/chat/completions
LLM_URL   = "https://api.groq.com/openai/v1/chat/completions"
LLM_MODEL = "openai/gpt-oss-120b"            # JUDGE + REPAIR
GEN_MODEL = "openai/gpt-oss-120b"        # ANSWER GENERATOR (different family from the judge)
LLM_MIN_INTERVAL = 2.1               # seconds between calls PER KEY (free tier ~30 req/min per account; paid accounts: try 0.3)

# JUDGE_MODE:  "all"   = LLM judge scores EVERY answer; score is used as a classifier feature (best F1)
#              "gated" = judge only the uncertain answers (cheaper; your proposal's confidence gate)
#              "off"   = no LLM judge
JUDGE_MODE      = "all"
JUDGE_BAND      = (0.30, 0.70)       # classifier probability inside this band = "uncertain" (used by the gated variant)
JUDGE_MAX_CALLS = 150                # cap for JUDGE_MODE = "gated"

REPAIR_ENABLED    = True       # old oracle repair OFF; Step 5 does detector-selected repair
REPAIR_MAX_N      = 30               # number of correctly-flagged hallucinated answers to repair
MAX_REPAIR_CYCLES = 2
N_GEN             = 250              # answers generated by GEN_MODEL in Step 5

OUTPUT_DIR = PROJECT_DIR / "achrp_outputs"
CACHE_DIR  = PROJECT_DIR / "achrp_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {'GPU' if HAS_GPU else 'CPU'} | records: {N_RECORDS} | NLI: {NLI_MODEL}")
print(f"judge: {LLM_MODEL} | generator: {GEN_MODEL} | judge mode: {JUDGE_MODE} | repair: {REPAIR_ENABLED} | folder: {PROJECT_DIR}")


Device: CPU | records: 250 | NLI: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
judge: openai/gpt-oss-120b | generator: openai/gpt-oss-120b | judge mode: all | repair: True | folder: c:\Users\Huawei\OneDrive\Desktop\New folder - Copy


In [4]:
# ============================================================
# CELL 3 — IMPORTS + DATASET LOADER
# ============================================================
import ast, json, re, time, hashlib, pickle, math
from collections import Counter
from getpass import getpass

import numpy as np
import pandas as pd
import requests
import spacy
from tqdm.auto import tqdm

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sentence_transformers import SentenceTransformer, CrossEncoder

REQUIRED_COLUMNS = ["Question", "Knowledge", "Ground Truth", "Hallucinated Answer"]
EXACT_NAMES = ["_pqa_labeled_train.csv", "_pqa_labeled_train_csv.xlsx", "_pqa_labeled_train.csv.xlsx",
               "pqa_labeled_train.csv", "pqa_labeled_train.xlsx", "medhallu.csv", "medhallu.xlsx"]

def find_dataset():
    if DATA_PATH_OVERRIDE:
        p = Path(DATA_PATH_OVERRIDE)
        if p.exists():
            return p
        raise FileNotFoundError(f"DATA_PATH_OVERRIDE does not exist: {p}")
    folders = [d for d in dict.fromkeys([PROJECT_DIR, Path.cwd(), Path("/content"), Path("/mnt/user-data/uploads")]) if d.exists()]
    for d in folders:                                           # 1) exact file names
        for name in EXACT_NAMES:
            if (d / name).exists():
                return d / name
    for d in folders:                                           # 2) any csv/xlsx that looks right
        files = sorted(list(d.glob("*.csv")) + list(d.glob("*.xlsx")))
        preferred = [p for p in files if any(k in p.name.lower() for k in ("pqa", "medhallu", "train"))]
        if preferred:
            return preferred[0]
    try:                                                        # 3) file picker (desktop only)
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk(); root.withdraw()
        chosen = filedialog.askopenfilename(title="Select your dataset",
                                            filetypes=[("CSV / Excel", "*.csv *.xlsx"), ("All files", "*.*")])
        root.destroy()
        if chosen:
            return Path(chosen)
    except Exception:
        pass
    raise FileNotFoundError("Dataset not found. Put it next to the notebook or set DATA_PATH_OVERRIDE in Cell 2.")

DATA_PATH = find_dataset()
print("Using dataset:", DATA_PATH)
df = pd.read_csv(DATA_PATH) if DATA_PATH.suffix.lower() == ".csv" else pd.read_excel(DATA_PATH)
df.columns = [str(c).strip() for c in df.columns]

missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns {missing}. Available: {list(df.columns)}")

df = df.sample(n=min(N_RECORDS, len(df)), random_state=RANDOM_SEED).copy()     # random sample, not head()
print(f"Dataset loaded: {len(df)} records -> {2 * len(df)} labelled answers.")


c:\Users\Huawei\OneDrive\Desktop\New folder - Copy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using dataset: c:\Users\Huawei\OneDrive\Desktop\New folder - Copy\_pqa_labeled_train.csv.xlsx
Dataset loaded: 250 records -> 500 labelled answers.


In [5]:
# CELL 4 — TEXT, CLAIM EXTRACTION AND CHUNKING
NLP = spacy.blank("en")
NLP.add_pipe("sentencizer")

def clean_text(x):
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return re.sub(r"\s+", " ", str(x).replace("\\n", " ").replace("\\t", " ")).strip()

def split_sentences(text):
    text = clean_text(text)
    if not text:
        return []
    return [s.text.strip() for s in NLP(text).sents if s.text.strip()]

def parse_knowledge(raw):
    if isinstance(raw, list):
        return [clean_text(x) for x in raw if clean_text(x)]
    raw = clean_text(raw)
    if not raw:
        return []
    if raw.startswith("[") and raw.endswith("]"):
        try:
            value = ast.literal_eval(raw)
            if isinstance(value, list):
                out = [clean_text(x) for x in value if clean_text(x)]
                if out:
                    return out
        except Exception:
            pass
    return [raw]

def finalize_claims(items):
    result, seen = [], set()
    for item in items:
        item = clean_text(item).strip(" ,;:-")
        if len(item.split()) < 4:
            continue
        key = item.lower()
        if key in seen:
            continue
        seen.add(key)
        if item[-1] not in ".!?":
            item += "."
        result.append(item)
        if len(result) >= 12:
            break
    return result

def extract_claims(answer):
    claims = []
    for sentence in split_sentences(answer):
        pieces = re.split(r"\s+(?:but|whereas|while)\s+", sentence, flags=re.IGNORECASE)
        if len(pieces) == 1:
            claims.append(sentence)
        elif all(len(p.split()) >= 4 for p in pieces):
            claims.extend(pieces)
        else:
            claims.append(sentence)
    return finalize_claims(claims)

def make_chunks(knowledge_sections):
    sentences, seen = [], set()
    for section in knowledge_sections:
        for sentence in split_sentences(section):
            key = sentence.lower()
            if key not in seen:
                seen.add(key)
                sentences.append(sentence)
    if not sentences:
        return []
    if len(sentences) <= CHUNK_SIZE:
        return [" ".join(sentences)]
    chunks, step = [], CHUNK_SIZE - CHUNK_OVERLAP
    for start in range(0, len(sentences), step):
        chunk = " ".join(sentences[start:start + CHUNK_SIZE]).strip()
        if chunk:
            chunks.append(chunk)
        if start + CHUNK_SIZE >= len(sentences):
            break
    return chunks

print("Claim extraction and chunking are ready.")


Claim extraction and chunking are ready.


In [6]:
# CELL 5 — LOCAL MODELS
DEVICE = "cuda" if HAS_GPU else "cpu"
NLI_BATCH = 32 if HAS_GPU else 16
torch.set_num_threads(os.cpu_count() or 4)
print("Device:", DEVICE, "| threads:", torch.get_num_threads())

embedding_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
nli_model = CrossEncoder(NLI_MODEL, device=DEVICE, max_length=512)

_cfg = getattr(nli_model, "config", None) or nli_model.model.config
id2label = {int(k): str(v).lower() for k, v in _cfg.id2label.items()}
ENTAIL_IDX = next(k for k, v in id2label.items() if "entail" in v)
CONTRA_IDX = next(k for k, v in id2label.items() if "contra" in v)
print("NLI labels:", id2label, "| models loaded.")


Device: cpu | threads: 8


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2846.39it/s]
c:\Users\Huawei\OneDrive\Desktop\New folder - Copy\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 740.60it/s]


NLI labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'} | models loaded.


In [7]:
# CELL 6 — HYBRID RETRIEVAL + NLI VERIFICATION + FEATURES
NLI_CACHE_FILE = CACHE_DIR / ("nli_" + hashlib.md5(NLI_MODEL.encode()).hexdigest()[:8] + ".pkl")
try:
    NLI_CACHE = pickle.load(open(NLI_CACHE_FILE, "rb")) if NLI_CACHE_FILE.exists() else {}
except Exception:
    NLI_CACHE = {}
print("NLI cache pairs:", len(NLI_CACHE))

def save_nli_cache():
    pickle.dump(NLI_CACHE, open(NLI_CACHE_FILE, "wb"))

def nli_probs(pairs):
    """pairs = [(premise, hypothesis)] -> array [n, 2] = (P_entail, P_contra). Cached on disk."""
    missing = list(dict.fromkeys(p for p in pairs if p not in NLI_CACHE))
    if missing:
        order = sorted(range(len(missing)), key=lambda i: len(missing[i][0]) + len(missing[i][1]))   # length-sorted = faster
        srt = [missing[i] for i in order]
        logits = np.asarray(nli_model.predict(srt, batch_size=NLI_BATCH, show_progress_bar=False), dtype=np.float32)
        if logits.ndim == 1:
            logits = logits[None, :]
        e = np.exp(logits - logits.max(1, keepdims=True))
        probs = e / e.sum(1, keepdims=True)
        for pair, pr in zip(srt, probs):
            NLI_CACHE[pair] = np.array([pr[ENTAIL_IDX], pr[CONTRA_IDX]], dtype=np.float32)
    return np.stack([NLI_CACHE[p] for p in pairs])

# ---- BM25 + Reciprocal Rank Fusion ----
_tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())

def bm25_scores(query, docs, k1=1.5, b=0.75):
    D, q = [_tok(d) for d in docs], set(_tok(query))
    N = len(D)
    avgdl = (sum(map(len, D)) / N) or 1.0
    df_ = Counter(t for d in D for t in set(d))
    out = np.zeros(N, dtype=np.float32)
    for i, d in enumerate(D):
        tf = Counter(d)
        for t in q:
            if t in tf:
                idf = math.log(1 + (N - df_[t] + 0.5) / (df_[t] + 0.5))
                out[i] += idf * tf[t] * (k1 + 1) / (tf[t] + k1 * (1 - b + b * len(d) / avgdl))
    return out

def top_chunks(claim, dense_row, chunks, k=None):
    k = k or TOP_K
    n = len(chunks)
    if k >= n:
        return list(range(n))
    bm = bm25_scores(claim, chunks)
    r_d = np.empty(n, int); r_d[np.argsort(-dense_row, kind="stable")] = np.arange(n)
    r_b = np.empty(n, int); r_b[np.argsort(-bm, kind="stable")] = np.arange(n)
    return [int(i) for i in np.argsort(-(1 / (60 + r_d) + 1 / (60 + r_b)), kind="stable")[:k]]

def knowledge_sentences(knowledge_sections):
    sents, seen = [], set()
    for sec in knowledge_sections:
        for s in split_sentences(sec):
            if s.lower() not in seen:
                seen.add(s.lower())
                sents.append(s)
    return sents

def full_context(knowledge_sections, max_words=None):
    words = " ".join(knowledge_sentences(knowledge_sections)).split()
    return " ".join(words[:(max_words or MAX_PREMISE_WORDS)])

def verify_claims(claims, chunks, full_text):
    if not claims:
        return []
    if not chunks:
        return [dict(claim=c, best_evidence="", evidence_text="", semantic=0.0, p_entail=0.0, max_contra=0.0,
                     status="unsupported") for c in claims]
    enc = lambda x: embedding_model.encode(x, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    sims = enc(claims) @ enc(chunks).T
    # pass 1 (cheap): top-k chunks only
    plan = [top_chunks(c, sims[i], chunks) for i, c in enumerate(claims)]
    prem = [list(dict.fromkeys(chunks[j] for j in idx)) for idx in plan]
    pr1 = nli_probs([(p, c) for i, c in enumerate(claims) for p in prem[i]])
    pos, need_full = 0, []
    for i, c in enumerate(claims):
        e1 = pr1[pos:pos + len(prem[i]), 0].max()
        pos += len(prem[i])
        if USE_FULL_CONTEXT and full_text and e1 < FULL_CONTEXT_IF_ENTAIL_BELOW and full_text not in prem[i]:
            need_full.append(i)
    # pass 2 (only where needed): whole abstract as the premise
    if need_full:
        nli_probs([(full_text, claims[i]) for i in need_full])
        for i in need_full:
            prem[i].append(full_text)
    probs = nli_probs([(p, c) for i, c in enumerate(claims) for p in prem[i]])
    out, pos = [], 0
    for i, c in enumerate(claims):
        idx, premises = plan[i], prem[i]
        pr = probs[pos:pos + len(premises)]
        pos += len(premises)
        ent, con = pr[:, 0], pr[:, 1]
        b = int(ent.argmax())
        sem = float(max(sims[i, j] for j in idx))
        e_max, c_max = float(ent.max()), float(con.max())
        status = ("contradicted" if (c_max >= CONTRADICTION_THRESHOLD and e_max < ENTAILMENT_THRESHOLD)
                  else "supported" if (e_max >= ENTAILMENT_THRESHOLD and sem >= SEMANTIC_THRESHOLD) else "unsupported")
        out.append(dict(claim=c, best_evidence=premises[b], evidence_text=" ".join(chunks[j] for j in sorted(idx)),
                        semantic=sem, p_entail=e_max, max_contra=c_max, status=status))
    return out

# ---- lexical "novelty": does the answer bring in terms/numbers the Knowledge never mentions? ----
_STOP = set("the a an of in on at to for and or but with by from as is are was were be been this that these those it its their there which who whom whose than then also not can may might could would should have has had does did into over under between among during after before about against through such more most less other some any each both few many much very".split())
def _terms(t):
    return {w[:6] for w in re.findall(r"[a-z][a-z\-]{3,}", t.lower()) if w not in _STOP}

def novelty(answer, knowledge_text):
    a, k = _terms(answer), _terms(knowledge_text)
    na, nk = set(re.findall(r"\d+(?:\.\d+)?", answer)), set(re.findall(r"\d+(?:\.\d+)?", knowledge_text))
    return len(a - k) / max(len(a), 1), float(len(na - nk))

# ---- direction / polarity cues (MedHallu hallucinations often flip "often" -> "rarely", "higher" -> "lower") ----
ANTONYMS = [("higher", "lower"), ("increase", "decrease"), ("increased", "decreased"), ("increases", "decreases"),
            ("more", "less"), ("better", "worse"), ("often", "rarely"), ("improve", "worsen"),
            ("positive", "negative"), ("effective", "ineffective"), ("significant", "insignificant"),
            ("greater", "smaller"), ("superior", "inferior"), ("associated", "unrelated"),
            ("beneficial", "harmful"), ("faster", "slower"), ("early", "late")]
NEG_WORDS   = {"no", "not", "never", "without", "neither", "nor", "fail", "failed", "lack", "absence", "unlikely", "cannot"}
HEDGE_WORDS = {"only", "solely", "exclusively", "primarily", "mainly", "always", "never", "all", "none", "rarely", "entirely"}
_words = lambda t: re.findall(r"[a-z]+", t.lower())

BASE_FEATURES = ["min_entail", "mean_entail", "max_contra", "mean_contra", "frac_supported", "mean_sem", "min_sem",
                 "ans_entail", "ans_contra", "novel_rate", "novel_num", "n_claims"]
EXTRA_FEATURES = ["s_ent_max", "s_ent_top2", "s_con_max", "flips", "neg_diff", "hedge_novel"]
FEATURES = BASE_FEATURES + EXTRA_FEATURES

def analyze_answer(answer, knowledge_sections):
    chunks, full = make_chunks(knowledge_sections), full_context(knowledge_sections)
    claims = extract_claims(answer) or ([clean_text(answer)] if clean_text(answer) else [])
    ver = verify_claims(claims, chunks, full)
    if not ver:
        return dict(**{f: 0.0 for f in BASE_FEATURES}, claims_json="[]")
    ent = np.array([v["p_entail"] for v in ver])
    con = np.array([v["max_contra"] for v in ver])
    sem = np.array([v["semantic"] for v in ver])
    ae, ac = (nli_probs([(full, clean_text(answer))])[0] if full else (0.0, 0.0))     # whole answer vs whole abstract
    nr, nn = novelty(answer, full)
    feats = dict(min_entail=ent.min(), mean_entail=ent.mean(), max_contra=con.max(), mean_contra=con.mean(),
                 frac_supported=np.mean([v["status"] == "supported" for v in ver]), mean_sem=sem.mean(), min_sem=sem.min(),
                 ans_entail=float(ae), ans_contra=float(ac), novel_rate=nr, novel_num=nn, n_claims=len(ver))
    return {**{k: float(v) for k, v in feats.items()}, "claims_json": json.dumps(ver, ensure_ascii=False)}

def extra_feats(answer, knowledge_sections):
    """Sentence-level NLI (whole answer vs every Knowledge sentence) + polarity / direction cues."""
    know_words, ans_words = set(_words(" ".join(knowledge_sections))), set(_words(answer))
    flips = sum(1 for a, b in ANTONYMS
                if (a in ans_words and a not in know_words and b in know_words)
                or (b in ans_words and b not in know_words and a in know_words))
    neg_diff = float(bool(ans_words & NEG_WORDS) and not (know_words & NEG_WORDS))
    hedge_novel = float(len((ans_words & HEDGE_WORDS) - know_words))
    sents, ans = knowledge_sentences(knowledge_sections), clean_text(answer)
    pr = nli_probs([(s, ans) for s in sents]) if (sents and ans) else np.zeros((1, 2), dtype=np.float32)
    return dict(s_ent_max=float(pr[:, 0].max()), s_ent_top2=float(np.sort(pr[:, 0])[-2:].mean()),
                s_con_max=float(pr[:, 1].max()), flips=float(flips), neg_diff=neg_diff, hedge_novel=hedge_novel)

def build_features(answer, knowledge_sections):
    out = analyze_answer(answer, knowledge_sections)
    out.update(extra_feats(answer, knowledge_sections))
    return out

print(f"Verification + feature extraction ready ({len(FEATURES)} features).")


NLI cache pairs: 11963
Verification + feature extraction ready (18 features).


In [8]:
# CELL 7 — LLM HELPER (multi-key rotation, thread-safe, cached), JUDGE SCORE, SELECTIVE REPAIR
import threading
from concurrent.futures import ThreadPoolExecutor

LLM_CACHE_FILE = CACHE_DIR / "llm_cache.json"
try:
    LLM_CACHE = json.load(open(LLM_CACHE_FILE, encoding="utf-8")) if LLM_CACHE_FILE.exists() else {}
except Exception:
    LLM_CACHE = {}
_unsaved = [0]
_lock = threading.Lock()

def save_llm_cache():
    with _lock:
        tmp = LLM_CACHE_FILE.with_suffix(".tmp")
        json.dump(LLM_CACHE, open(tmp, "w", encoding="utf-8"), ensure_ascii=False)
        os.replace(tmp, LLM_CACHE_FILE)
        _unsaved[0] = 0

def load_keys():
    keys = [k.strip() for k in os.getenv("GROQ_API_KEYS", "").split(",")]
    keys += [os.getenv(f"GROQ_API_KEY_{i}", "").strip() for i in range(1, 10)]
    keys += [os.getenv("GROQ_API_KEY", "").strip(), os.getenv("LLM_API_KEY", "").strip()]
    return list(dict.fromkeys(k for k in keys if k.startswith("gsk_")))

KEYS = load_keys()
assert KEYS, "No gsk_ keys found. Put GROQ_API_KEYS=gsk_a,gsk_b,... in .env and run Cell 0 first."
print("Keys loaded:", len(KEYS))
_next_free = {k: 0.0 for k in KEYS}          # earliest time each key may be used again

def _acquire_key():
    """Round-robin by availability: always take the key that has been idle longest."""
    while True:
        with _lock:
            k = min(KEYS, key=lambda x: _next_free[x])
            now = time.time()
            if _next_free[k] <= now:
                _next_free[k] = now + LLM_MIN_INTERVAL
                return k
            wait = _next_free[k] - now
        time.sleep(min(wait, 0.5))

def llm_chat(system, user, max_tokens=200, model=None):
    model = model or LLM_MODEL
    cid = hashlib.sha256(json.dumps([LLM_URL, model, system, user, max_tokens], ensure_ascii=False).encode()).hexdigest()
    if cid in LLM_CACHE:
        return LLM_CACHE[cid]
    body = {"model": model, "temperature": 0.0, "max_tokens": max_tokens,
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
    if "gpt-oss" in model:
        body["reasoning_effort"] = "low"
    last_err = None
    for attempt in range(8):
        key = _acquire_key()
        try:
            r = requests.post(LLM_URL, headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
                              json=body, timeout=90)
        except requests.exceptions.RequestException as e:
            last_err = e
            time.sleep(2 ** min(attempt, 5))
            continue
        if r.status_code == 429:                                   # rate limited: park ONLY this key
            try:
                wait = float(r.headers.get("retry-after", 10))
            except (TypeError, ValueError):
                wait = 10.0
            with _lock:
                _next_free[key] = time.time() + wait + 1
            continue
        if r.status_code in (401, 403):                            # bad key: stop using it
            with _lock:
                _next_free[key] = time.time() + 1e9
                if all(_next_free[k] > time.time() + 1e8 for k in KEYS):
                    raise RuntimeError("All keys rejected: " + r.text[:200])
            continue
        if r.status_code in (400, 404):
            raise RuntimeError(f"LLM error {r.status_code} | model={model} | {r.text[:250]}")
        if r.status_code >= 500:
            time.sleep(2 ** min(attempt, 5))
            continue
        r.raise_for_status()
        txt = (r.json()["choices"][0]["message"].get("content") or "").strip()
        if txt:
            with _lock:
                LLM_CACHE[cid] = txt
                _unsaved[0] += 1
                need_save = _unsaved[0] >= 10
            if need_save:
                save_llm_cache()
        return txt
    raise RuntimeError(f"LLM call failed after retries ({last_err})")

def llm_preflight():
    try:
        ok = bool(llm_chat("You are a connection test.", "Reply with the single word OK.", max_tokens=300))
        if not ok:
            print("LLM preflight: empty reply (model may need more tokens).")
        return ok
    except Exception as e:
        print("LLM PREFLIGHT FAILED ->", type(e).__name__, ":", str(e)[:300])
        return False

# ---------------- judge (unchanged prompt; only the model argument is new) ----------------
JUDGE_SYSTEM = "You are a strict biomedical fact-checker. You compare an answer with a source passage."
JUDGE_PROMPT = """Question: {question}

Source passage:
{passage}

Answer to check:
{answer}

Decide whether the answer is faithful to the source passage.
- SUPPORTED: everything the answer states is in the passage, or is a reasonable summary / conclusion of it.
- HALLUCINATED: the answer contradicts the passage, or asserts specific mechanisms, entities, numbers, directions of effect or causal links that the passage does not state.
Typical hallucination types: misreading the question, claiming more or less than the passage supports, wrong mechanism or cause, invented methods or numbers, reversed direction of effect (for example "often" vs "rarely", "higher" vs "lower").
Reply with JSON only: {{"label": "SUPPORTED" or "HALLUCINATED", "confidence": integer from 50 to 100}}"""

def judge_p(question, answer, knowledge_sections, model=None):
    passage = full_context(knowledge_sections, max_words=900)
    prompt = JUDGE_PROMPT.format(question=question, passage=passage, answer=answer)
    for max_tokens in (600, 1800):
        txt = llm_chat(JUDGE_SYSTEM, prompt, max_tokens=max_tokens, model=model).upper()
        lab = re.search(r'LABEL"?\s*:\s*"?(SUPPORTED|HALLUCINATED)', txt) or re.search(r"(SUPPORTED|HALLUCINATED)", txt)
        if lab:
            con = re.search(r'CONFIDENCE"?\s*:\s*"?(\d+)', txt)
            c = min(max(int(con.group(1)), 0), 100) / 100 if con else 0.8
            return c if lab.group(1) == "HALLUCINATED" else 1.0 - c
    return None

# ---------------- selective repair (unchanged) ----------------
REPAIR_SYSTEM = "You are a biomedical factual repair component. Use only the supplied evidence."
REPAIR_PROMPT = """Repair this claim using ONLY the evidence.

Claim: {claim}
Status: {status}

Evidence:
{evidence}

Rules:
1. Change only what is unsupported or contradicted; keep correct information.
2. Do not add anything the evidence does not state.
3. Return exactly one factual sentence, or exactly DELETE if the evidence says nothing about the claim.
4. Do not mention evidence, verification or hallucination."""

NO_ANSWER = "The available evidence does not support a definitive answer to this question."

def repair_answer(answer, knowledge_sections):
    chunks, full = make_chunks(knowledge_sections), full_context(knowledge_sections)
    claims = extract_claims(answer) or [clean_text(answer)]
    state = [dict(text=v["claim"], status=v["status"], evidence=v["evidence_text"] or v["best_evidence"])
             for v in verify_claims(claims, chunks, full)]
    log = []
    for _ in range(MAX_REPAIR_CYCLES):
        todo = [s for s in state if s["status"] in ("unsupported", "contradicted")]
        if not todo:
            break
        for s in todo:
            raw = clean_text(llm_chat(REPAIR_SYSTEM, REPAIR_PROMPT.format(claim=s["text"], status=s["status"],
                                                                         evidence=s["evidence"]), max_tokens=500))
            new = raw.strip()
            if not new or new.upper().startswith("DELETE"):
                s["status"] = "deleted"
                continue
            if new[-1] not in ".!?":
                new += "."
            v = verify_claims([new], chunks, full)[0]
            log.append((s["text"], new, v["status"]))
            s.update(text=new, status=v["status"], evidence=v["evidence_text"] or v["best_evidence"])
    kept = [s["text"] for s in state if s["status"] == "supported"]
    return (" ".join(kept) if kept else NO_ANSWER), log

print("LLM helper (multi-key), judge and repair are ready.")


Keys loaded: 5
LLM helper (multi-key), judge and repair are ready.


## Step 1 — analyse all answers (checkpointed)

In [9]:
# CELL 9 — RUN ON ALL RECORDS  (ground-truth answer = label 0, hallucinated answer = label 1)
# Checkpointed: safe to stop and re-run; NLI results are cached on disk.
FEAT_FILE = OUTPUT_DIR / "achrp_v3_features.pkl"
try:
    done = pickle.load(open(FEAT_FILE, "rb")) if FEAT_FILE.exists() else {}
except Exception:
    done = {}
t0 = time.time()

for n, (rid, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc="records")):
    know, question = parse_knowledge(row["Knowledge"]), clean_text(row["Question"])
    for atype, col, label in (("ground_truth", "Ground Truth", 0), ("hallucinated", "Hallucinated Answer", 1)):
        key = f"{int(rid)}|{atype}"
        if key in done and all(f in done[key] for f in FEATURES):
            continue
        ans = clean_text(row[col])
        done[key] = dict(record_id=int(rid), answer_type=atype, label=label, question=question, answer=ans,
                         **build_features(ans, know))
    if (n + 1) % 5 == 0:
        pickle.dump(done, open(FEAT_FILE, "wb"))
        save_nli_cache()

pickle.dump(done, open(FEAT_FILE, "wb"))
save_nli_cache()

selected = set(int(i) for i in df.index)
res = pd.DataFrame([v for v in done.values() if v["record_id"] in selected])
res = res.sort_values(["record_id", "label"]).reset_index(drop=True)
k_by_rid = {int(r): parse_knowledge(df.loc[r, "Knowledge"]) for r in df.index}
print(f"{len(res)} labelled answers | {time.time() - t0:.0f}s this run")
res[["answer_type", "label"] + FEATURES].groupby("answer_type").mean().round(3).T


records:   0%|          | 0/250 [00:00<?, ?it/s]

records: 100%|██████████| 250/250 [00:05<00:00, 45.12it/s]


500 labelled answers | 6s this run


answer_type,ground_truth,hallucinated
label,0.000,1.000
min_entail,0.251,0.147
mean_entail,0.467,0.183
max_contra,0.173,0.335
mean_contra,0.095,0.308
frac_supported,0.465,0.159
mean_sem,0.658,0.715
min_sem,0.566,0.684
ans_entail,0.411,0.168
ans_contra,0.057,0.241


## Step 2 — honest evaluation without the LLM (grouped + nested CV)

In [11]:
# CELL 10 — CLASSIFIER (grouped, nested cross-validation) + NO-LLM BASELINES
# A record's two answers are never split between train and test (GroupKFold).
# The decision threshold is also chosen only on the training folds (nested), so it does not leak.
def metrics_of(y, pred, score):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(accuracy=accuracy_score(y, pred), precision=precision_score(y, pred, zero_division=0),
                recall=recall_score(y, pred, zero_division=0), f1=f1_score(y, pred, zero_division=0),
                auroc=roc_auc_score(y, score) if len(set(y)) == 2 else float("nan"), TN=tn, FP=fp, FN=fn, TP=tp)

def boot_ci(y, pred, groups, metric, n=1000, seed=0):
    """Bootstrap over RECORDS (both answers of a record move together)."""
    y, pred, groups = np.asarray(y), np.asarray(pred), np.asarray(groups)
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    idx_by = {g: np.where(groups == g)[0] for g in uniq}
    vals = []
    for _ in range(n):
        ii = np.concatenate([idx_by[g] for g in rng.choice(uniq, len(uniq), replace=True)])
        vals.append(metric(y[ii], pred[ii]))
    return np.percentile(vals, [2.5, 97.5])

def _gkf(n_groups, k):
    return GroupKFold(n_splits=max(2, min(k, n_groups)))

THRESHOLDS = np.linspace(0.2, 0.8, 25)

def oof_stack(make_model, X, y, groups, folds=5):
    """Out-of-fold probabilities + a per-row threshold chosen (to maximise F1) on the training folds only."""
    p, thr = np.zeros(len(y)), np.full(len(y), 0.5)
    for tr, te in _gkf(len(set(groups)), folds).split(X, y, groups):
        inner = np.zeros(len(tr))
        for itr, ite in _gkf(len(set(groups[tr])), 4).split(X[tr], y[tr], groups[tr]):
            inner[ite] = make_model().fit(X[tr][itr], y[tr][itr]).predict_proba(X[tr][ite])[:, 1]
        f1s = [f1_score(y[tr], (inner >= s).astype(int), zero_division=0) for s in THRESHOLDS]
        thr[te] = max(zip(f1s, THRESHOLDS), key=lambda a: (round(a[0], 4), -abs(a[1] - 0.5)))[1]
        p[te] = make_model().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return p, thr

mk_lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=5000))
mk_gb = lambda: GradientBoostingClassifier(n_estimators=120, max_depth=2, learning_rate=0.08, random_state=RANDOM_SEED)

assert res["record_id"].nunique() >= 10, "Use at least ~10 records for a meaningful cross-validation."
y, groups = res["label"].values, res["record_id"].values
X_nli = res[FEATURES].values

TABLE = {}
base_pred = (res["frac_supported"] < 1.0).astype(int).values                      # your old rule
TABLE["old rule (any claim unsupported)"] = metrics_of(y, base_pred, 1 - res["frac_supported"].values)

p_nli, thr_nli = oof_stack(mk_lr, X_nli, y, groups)
p_gb, _ = oof_stack(mk_gb, X_nli, y, groups)
res["p_nli"] = p_nli
TABLE["NLI features + gradient boosting (thr 0.5)"] = metrics_of(y, (p_gb >= 0.5).astype(int), p_gb)
TABLE["NLI features + LR (tuned thr)"]             = metrics_of(y, (p_nli >= thr_nli).astype(int), p_nli)
TABLE["NLI features + LR (thr 0.5)"]               = metrics_of(y, (p_nli >= 0.5).astype(int), p_nli)


print(pd.DataFrame(TABLE).T.round(3).to_string())
final_model_nli = mk_lr().fit(X_nli, y)                                           # used later for new answers (repair)
print("\nFeature weights (standardised, LR):")
print(pd.Series(final_model_nli[-1].coef_[0], index=FEATURES).sort_values().round(2).to_string())


                                            accuracy  precision  recall     f1  auroc     TN     FP    FN     TP
old rule (any claim unsupported)               0.560      0.537   0.880  0.667  0.736   60.0  190.0  30.0  220.0
NLI features + gradient boosting (thr 0.5)     0.760      0.760   0.760  0.760  0.849  190.0   60.0  60.0  190.0
NLI features + LR (tuned thr)                  0.766      0.742   0.816  0.777  0.839  179.0   71.0  46.0  204.0
NLI features + LR (thr 0.5)                    0.782      0.781   0.784  0.782  0.839  195.0   55.0  54.0  196.0

Feature weights (standardised, LR):
mean_entail      -1.14
n_claims         -0.60
max_contra       -0.28
s_ent_max        -0.14
frac_supported   -0.12
ans_entail       -0.02
neg_diff         -0.01
mean_sem          0.10
min_sem           0.10
s_ent_top2        0.10
hedge_novel       0.11
flips             0.12
s_con_max         0.15
novel_rate        0.19
novel_num         0.25
ans_contra        0.49
mean_contra       0.59
min_ent

## Step 3 — LLM judge and final systems (with vs without LLM)

In [12]:
# CELL 11 — LLM JUDGE + FINAL SYSTEMS (with vs without LLM)
res["judge_p"] = np.nan
JUDGE_ON = False
if JUDGE_MODE != "off":
    JUDGE_ON = llm_preflight()
    if not JUDGE_ON:
        print("-> continuing WITHOUT the LLM judge (fix the key/model in Cell 2 and re-run this cell to enable it).")

if JUDGE_ON:
    if JUDGE_MODE == "all":
        todo = list(res.index)
    else:                                                                          # "gated": uncertain answers only
        lo_b, hi_b = JUDGE_BAND
        cand = res.index[res["p_nli"].between(lo_b, hi_b)]
        todo = sorted(cand, key=lambda i: abs(res.loc[i, "p_nli"] - 0.5))[:JUDGE_MAX_CALLS]
    print(f"Judging {len(todo)} of {len(res)} answers with {LLM_MODEL} (cached calls are free; safe to re-run) ...")
    failures, first_err = 0, [None]

    def _judge_row(i):
        r = res.loc[i]
        try:
            return i, judge_p(r["question"], r["answer"], k_by_rid[int(r["record_id"])])
        except Exception as e:
            if first_err[0] is None:
                first_err[0] = str(e)[:200]
            return i, None

    try:
        with ThreadPoolExecutor(max_workers=min(2 * len(KEYS), 8)) as ex:
            for i, v in tqdm(ex.map(_judge_row, todo), total=len(todo), desc="LLM judge"):
                if v is None:
                    failures += 1
                else:
                    res.loc[i, "judge_p"] = v
    finally:
        save_llm_cache()
    print(f"Judge scored {int(res['judge_p'].notna().sum())} answers ({failures} failures).")
    if first_err[0]:
        print("first error:", first_err[0], "| re-run this cell to retry the missing ones (cached calls are free)")

coverage   = float(res["judge_p"].notna().mean())
have_judge = JUDGE_ON and coverage > 0                                             # at least some answers were judged
judge_all  = have_judge and JUDGE_MODE == "all" and coverage >= 0.9                # full coverage -> stacking possible
if JUDGE_ON and JUDGE_MODE == "all" and not judge_all:
    print(f"WARNING: judge coverage is only {coverage:.0%} (< 90%), so 'judge score as a feature' is disabled. "
          "Re-run this cell: cached judgements are free and the missing ones will be retried.")

FINAL = dict(TABLE)
if have_judge:
    lo_b, hi_b = JUDGE_BAND
    use_j = (res["p_nli"].between(lo_b, hi_b) & res["judge_p"].notna()).values
    s_gate = np.where(use_j, res["judge_p"].fillna(0.5).values, res["p_nli"].values)
    FINAL["+ confidence-gated LLM judge (thr 0.5)"] = metrics_of(y, (s_gate >= 0.5).astype(int), s_gate)
    print(f"Gate sent {int(use_j.sum())} of {len(res)} answers ({use_j.mean():.0%}) to the judge.")
if judge_all:
    jf = res["judge_p"].fillna(res["p_nli"]).values
    FINAL["LLM judge alone (thr 0.5)"] = metrics_of(y, (jf >= 0.5).astype(int), jf)
    res["judge_f"] = jf
    FEATS_S = FEATURES + ["judge_f"]
    X_s = res[FEATS_S].values
    p_s, thr_s = oof_stack(mk_lr, X_s, y, groups)
    res["p_stack"] = p_s
    FINAL["NLI features + judge score (tuned thr) "] = metrics_of(y, (p_s >= thr_s).astype(int), p_s)
    FINAL["NLI features + judge score (thr 0.5)  <- ACHRP full"]       = metrics_of(y, (p_s >= 0.5).astype(int), p_s)
    

# ---- declare the final system BEFORE looking at test numbers: stacked+tuned > gated > NLI-only tuned ----
if judge_all:
    FINAL_NAME, p_fin, pred_fin = "NLI features + judge score (tuned thr)", p_s, (p_s >= thr_s).astype(int)
    FINAL_FEATS, X_fin, T_FINAL = FEATS_S, X_s, float(np.median(thr_s))
elif have_judge:
    FINAL_NAME, p_fin, pred_fin = "NLI features + confidence-gated LLM judge", s_gate, (s_gate >= 0.5).astype(int)
    FINAL_FEATS, X_fin, T_FINAL = FEATURES, X_nli, float(np.median(thr_nli))
else:
    FINAL_NAME, p_fin, pred_fin = "NLI features + LR (tuned thr)", p_nli, (p_nli >= thr_nli).astype(int)
    FINAL_FEATS, X_fin, T_FINAL = FEATURES, X_nli, float(np.median(thr_nli))

res["p_final"], res["pred_final"] = p_fin, pred_fin
final_model = mk_lr().fit(X_fin, y)
res["p_insample"] = final_model.predict_proba(X_fin)[:, 1]

print("\n" + "=" * 100)
print("DETECTION RESULTS (hallucinated = positive class)  -  n =", len(y), "answers /", res["record_id"].nunique(), "records")
print(pd.DataFrame(FINAL).T.round(3).to_string())
fm = metrics_of(y, pred_fin, p_fin)
lo_f, hi_f = boot_ci(y, pred_fin, groups, lambda a, b: f1_score(a, b, zero_division=0))
lo_a, hi_a = boot_ci(y, pred_fin, groups, accuracy_score)
print(f"\nFINAL SYSTEM: {FINAL_NAME}")
print(f"  accuracy {fm['accuracy']:.3f} [{lo_a:.3f}, {hi_a:.3f}] | precision {fm['precision']:.3f} | recall {fm['recall']:.3f} | "
      f"F1 {fm['f1']:.3f} [{lo_f:.3f}, {hi_f:.3f}]   (95% record-level bootstrap CI)")


Judging 500 of 500 answers with openai/gpt-oss-120b (cached calls are free; safe to re-run) ...


LLM judge: 100%|██████████| 500/500 [00:00<00:00, 5396.56it/s]


Judge scored 500 answers (0 failures).
Gate sent 140 of 500 answers (28%) to the judge.

DETECTION RESULTS (hallucinated = positive class)  -  n = 500 answers / 250 records
                                                     accuracy  precision  recall     f1  auroc     TN     FP    FN     TP
old rule (any claim unsupported)                        0.560      0.537   0.880  0.667  0.736   60.0  190.0  30.0  220.0
NLI features + gradient boosting (thr 0.5)              0.760      0.760   0.760  0.760  0.849  190.0   60.0  60.0  190.0
NLI features + LR (tuned thr)                           0.766      0.742   0.816  0.777  0.839  179.0   71.0  46.0  204.0
NLI features + LR (thr 0.5)                             0.782      0.781   0.784  0.782  0.839  195.0   55.0  54.0  196.0
+ confidence-gated LLM judge (thr 0.5)                  0.834      0.813   0.868  0.839  0.857  200.0   50.0  33.0  217.0
LLM judge alone (thr 0.5)                               0.842      0.779   0.956  0.858  0.929 

## Step 4 — old oracle repair (OFF in v5; Step 5 does detector-selected repair)

In [13]:
# CELL 12 — SELECTIVE REPAIR + RE-VERIFICATION (hallucination rate before vs after)
def score_new(question, text, knowledge_sections):
    """P(hallucinated) for a NEW answer (e.g. a repaired one) with the same pipeline as the evaluation."""
    f = build_features(text, knowledge_sections)
    p_n = float(final_model_nli.predict_proba(np.array([[f[k] for k in FEATURES]]))[0, 1])
    if "judge_f" in FINAL_FEATS:
        jp = judge_p(question, text, knowledge_sections)
        f["judge_f"] = p_n if jp is None else jp
        return float(final_model.predict_proba(np.array([[f[k] for k in FINAL_FEATS]]))[0, 1])
    return p_n

def unsupported_rate(ver):
    return float(np.mean([v["status"] != "supported" for v in ver])) if ver else 0.0

REPAIR_TABLE = None
if not REPAIR_ENABLED:
    print("Repair is OFF (REPAIR_ENABLED = False in Cell 2).")
elif not llm_preflight():
    print("Repair skipped: the LLM is not reachable (see message above).")
else:
    flagged = res[(res["label"] == 1) & (res["pred_final"] == 1)].head(REPAIR_MAX_N)
    rows = []
    for i, r in tqdm(flagged.iterrows(), total=len(flagged), desc="repair"):
        know = k_by_rid[int(r["record_id"])]
        try:
            fixed, log = repair_answer(r["answer"], know)
        except Exception as e:
            print("repair failed for one answer:", str(e)[:160])
            continue
        abstained = (fixed == NO_ANSWER)
        before_claims = unsupported_rate(json.loads(r["claims_json"]))
        if abstained:
            after_claims, p_after = 0.0, 0.0
        else:
            ver = verify_claims(extract_claims(fixed), make_chunks(know), full_context(know))
            after_claims, p_after = unsupported_rate(ver), score_new(r["question"], fixed, know)
        rows.append(dict(record_id=int(r["record_id"]), question=r["question"][:70], before=r["answer"][:120], after=fixed[:120],
                         abstained=abstained, p_halluc_before=round(float(r["p_insample"]), 3), p_halluc_after=round(p_after, 3),
                         claim_unsupported_before=round(before_claims, 3), claim_unsupported_after=round(after_claims, 3)))
    save_llm_cache(); save_nli_cache()
    if rows:
        rep = pd.DataFrame(rows)
        REPAIR_TABLE = rep
        flag_before = float((rep["p_halluc_before"] >= T_FINAL).mean())
        flag_after  = float((rep["p_halluc_after"] >= T_FINAL).mean())
        summary = pd.DataFrame({
            "metric": ["answers repaired", "answers still flagged as hallucinated (before)", "answers still flagged as hallucinated (after)",
                       "answers replaced by 'no definitive answer'", "mean P(hallucinated) before", "mean P(hallucinated) after",
                       "unsupported-claim rate before", "unsupported-claim rate after"],
            "value": [len(rep), f"{flag_before:.1%}", f"{flag_after:.1%}", f"{int(rep['abstained'].sum())} ({rep['abstained'].mean():.0%})",
                      round(rep["p_halluc_before"].mean(), 3), round(rep["p_halluc_after"].mean(), 3),
                      f"{rep['claim_unsupported_before'].mean():.1%}", f"{rep['claim_unsupported_after'].mean():.1%}"]})
        print(summary.to_string(index=False))
        print("\nNote: 'before' scores use the final model fitted on all answers (in-sample), so before/after are comparable;")
        print("abstentions count as 'not hallucinated' - report them separately in the paper.\n")
        print(rep[["question", "before", "after", "p_halluc_before", "p_halluc_after"]].head(10).to_string(index=False))
    else:
        print("No answers were repaired (nothing flagged as a true positive).")


repair: 100%|██████████| 30/30 [00:09<00:00,  3.24it/s]

                                        metric   value
                              answers repaired      30
answers still flagged as hallucinated (before)  100.0%
 answers still flagged as hallucinated (after)   10.0%
    answers replaced by 'no definitive answer' 9 (30%)
                   mean P(hallucinated) before   0.913
                    mean P(hallucinated) after   0.132
                 unsupported-claim rate before   96.7%
                  unsupported-claim rate after    0.0%

Note: 'before' scores use the final model fitted on all answers (in-sample), so before/after are comparable;
abstentions count as 'not hallucinated' - report them separately in the paper.

                                                              question                                                                                                                   before                                                                                                                    after  p

## Step 5 — LLM generates answers → ACHRP detects → ACHRP repairs (detector-selected, no labels used)

In [14]:
# CELL 14 — LLM (GEN_MODEL) GENERATES ANSWERS -> ACHRP DETECTS -> ACHRP REPAIRS (detector-selected, no labels used)
# N_GEN and GEN_MODEL are set in CELL 2. Needs final_model, final_model_nli, FINAL_FEATS, T_FINAL from the earlier cells.
GEN_SYSTEM = "You are a biomedical assistant. Answer the question in 1-2 sentences."

full = pd.read_csv(DATA_PATH) if DATA_PATH.suffix.lower() == ".csv" else pd.read_excel(DATA_PATH)
full.columns = [str(c).strip() for c in full.columns]
gen_df = full.drop(index=df.index, errors="ignore").sample(n=N_GEN, random_state=RANDOM_SEED)   # held-out records

# 1) generate in parallel (closed-book, temperature 0)
def _gen(item):
    rid, r = item
    try:
        return int(rid), clean_text(llm_chat(GEN_SYSTEM, clean_text(r["Question"]), max_tokens=800, model=GEN_MODEL))
    except Exception as e:
        print("generation failed:", str(e)[:120])
        return int(rid), ""

with ThreadPoolExecutor(max_workers=min(2 * len(KEYS), 8)) as ex:
    gen_answers = dict(tqdm(ex.map(_gen, gen_df.iterrows()), total=len(gen_df), desc="generate"))
save_llm_cache()

# 2) warm the judge cache in parallel so detection below does not wait on the API
def _warm(item):
    rid, r = item
    a = gen_answers.get(int(rid), "")
    if a:
        try:
            judge_p(clean_text(r["Question"]), a, parse_knowledge(r["Knowledge"]))
        except Exception:
            pass

with ThreadPoolExecutor(max_workers=min(2 * len(KEYS), 8)) as ex:
    list(tqdm(ex.map(_warm, gen_df.iterrows()), total=len(gen_df), desc="judge (parallel)"))
save_llm_cache()

# 3) detect (local NLI is sequential; judge calls now hit the cache)
rows = []
for rid, r in tqdm(gen_df.iterrows(), total=len(gen_df), desc="detect"):
    q, know = clean_text(r["Question"]), parse_knowledge(r["Knowledge"])
    ans = gen_answers.get(int(rid), "")
    if not ans:
        continue
    p = score_new(q, ans, know)
    ver = verify_claims(extract_claims(ans), make_chunks(know), full_context(know))
    rows.append(dict(record_id=int(rid), question=q, answer=ans, gold_answer=clean_text(r["Ground Truth"]),
                     p_before=p, flagged=p >= T_FINAL, unsup_before=unsupported_rate(ver), know=know))
save_nli_cache()
gen = pd.DataFrame(rows)
print(f"Generated {len(gen)} answers with {GEN_MODEL} | flagged as hallucinated: {gen['flagged'].sum()} ({gen['flagged'].mean():.0%})")

# 4) repair only what the detector flagged
rep_rows = []
for i, r in tqdm(gen[gen["flagged"]].iterrows(), total=int(gen["flagged"].sum()), desc="repair"):
    try:
        fixed, _ = repair_answer(r["answer"], r["know"])
    except Exception as e:
        print("repair failed:", str(e)[:120])
        continue
    abst = (fixed == NO_ANSWER)
    if abst:
        p_after, unsup_after = 0.0, 0.0
    else:
        v = verify_claims(extract_claims(fixed), make_chunks(r["know"]), full_context(r["know"]))
        p_after, unsup_after = score_new(r["question"], fixed, r["know"]), unsupported_rate(v)
    rep_rows.append(dict(record_id=r["record_id"], question=r["question"][:80], before=r["answer"], after=fixed,
                         abstained=abst, p_before=r["p_before"], p_after=p_after,
                         unsup_before=r["unsup_before"], unsup_after=unsup_after))
save_llm_cache(); save_nli_cache()

rep = pd.DataFrame(rep_rows)
if len(rep):
    print(f"\nRepaired {len(rep)} flagged answers | abstentions: {int(rep['abstained'].sum())} ({rep['abstained'].mean():.0%})")
    print(f"still flagged after repair: {(rep['p_after'] >= T_FINAL).mean():.1%}  (before: 100%)")
    print(f"mean P(halluc): {rep['p_before'].mean():.3f} -> {rep['p_after'].mean():.3f}")
    print(f"unsupported-claim rate: {rep['unsup_before'].mean():.1%} -> {rep['unsup_after'].mean():.1%}")
    nonabs = rep[~rep["abstained"]]
    if len(nonabs):
        print(f"excluding abstentions: still flagged {(nonabs['p_after'] >= T_FINAL).mean():.1%}, "
              f"mean P {nonabs['p_before'].mean():.3f} -> {nonabs['p_after'].mean():.3f}")
print("NOTE: flag rates here are detector outputs only (no manual accuracy check yet) - report them as unverified.")

gen.drop(columns=["know"]).to_csv(OUTPUT_DIR / "achrp_v5_generated_answers.csv", index=False)
rep.to_csv(OUTPUT_DIR / "achrp_v5_generated_repair.csv", index=False)


detect: 100%|██████████| 250/250 [3:07:02<00:00, 44.89s/it]   


Generated 250 answers with openai/gpt-oss-120b | flagged as hallucinated: 164 (66%)


repair: 100%|██████████| 164/164 [36:03<00:00, 13.19s/it] 


Repaired 164 flagged answers | abstentions: 65 (40%)
still flagged after repair: 16.5%  (before: 100%)
mean P(halluc): 0.846 -> 0.176
unsupported-claim rate: 85.7% -> 0.5%
excluding abstentions: still flagged 27.3%, mean P 0.844 -> 0.291
NOTE: flag rates here are detector outputs only (no manual accuracy check yet) - report them as unverified.


## Step 6 — save

In [17]:
# CELL 13 — SAVE
res.drop(columns=["claims_json"]).to_csv(OUTPUT_DIR / "achrp_v3_predictions.csv", index=False)
res[["record_id", "answer_type", "label", "p_nli", "judge_p", "p_final", "pred_final", "claims_json"]].to_csv(
    OUTPUT_DIR / "achrp_v3_claims.csv", index=False)
pd.DataFrame(FINAL).T.round(4).to_csv(OUTPUT_DIR / "achrp_v3_detection_table.csv")
json.dump({k: {m: float(v) for m, v in d.items()} for k, d in FINAL.items()},
          open(OUTPUT_DIR / "achrp_v3_metrics.json", "w"), indent=2)
if REPAIR_TABLE is not None:
    REPAIR_TABLE.to_csv(OUTPUT_DIR / "achrp_v3_repair.csv", index=False)
save_llm_cache(); save_nli_cache()
print("Saved to", OUTPUT_DIR)


Saved to c:\Users\Huawei\OneDrive\Desktop\New folder - Copy\achrp_outputs
